# 03 · Does discounting move demand?
Event study: review velocity 7 days before vs after each discount change. **Correlational, and review velocity is a proxy for sales** — see README limitations.

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from pricing.db import get_engine, load_snapshots, load_scrape_log
from pricing.clean import clean_snapshots, dim_products, gap_report
from pricing import analysis as an
from pricing.export import load_matches
raw = load_snapshots(get_engine())
clean, report = clean_snapshots(raw)
events = an.discount_events(clean)
print(f"{len(raw):,} raw rows -> {len(clean):,} clean rows, {clean.product_url.nunique()} products")

In [ ]:
es = an.build_event_study(clean, events)
es["brand_tier"] = es.brand.map(an.brand_tiers(clean)) if len(es) else None
deeper = es[es.discount_change > 0] if len(es) else es
print(len(es), "eligible events;", len(deeper), "deeper-discount events;", int((~deeper.overlap).sum()) if len(deeper) else 0, "with no overlapping change")

In [ ]:
fit = an.elasticity(deeper); fit

In [ ]:
an.discount_floor(deeper)

In [ ]:
if len(deeper):
    ax = deeper.plot.scatter("discount_pct", "velocity_change", figsize=(8, 4), alpha=.6)
    if fit.get("available"):
        xs = np.array([fit["x_min"], fit["x_max"]]); ax.plot(xs, fit["intercept"] + fit["slope"] * xs, "k-")

In [ ]:
pd.DataFrame(an.lift_by_segment(deeper, "price_band")), pd.DataFrame(an.lift_by_segment(deeper, "brand_tier"))

In [ ]:
an.margin_erosion_flags(deeper)[["product_url", "brand", "event_date", "discount_pct", "velocity_before", "velocity_after"]]